# WHW06 starter notebook: how good is $\sin\theta \approx \theta$?

The pendulum equation

$$\theta'' = -\frac{g}{L}\sin\theta$$

only becomes the simple harmonic oscillator $\theta'' = -\frac{g}{L}\,\theta$ because
$\sin\theta \approx \theta$ for small angles. That's a **linear approximation**: near
$\theta = 0$, the graph of $\sin\theta$ looks like its tangent line, which is just $\theta$.
But how small does "small" have to be? We're going to find out, and then we're going to find out
*how fast* the approximation gets worse as $\theta$ grows.

**New Python in this notebook:**

- **NumPy arrays, element-wise.** `np.sin(theta)` on an array of angles gives an array of
  sines, one per angle. Arithmetic like `theta - np.sin(theta)` also works one element at a time.
  (This is like a Listable function in Mathematica.)
- **Log-log plots** with `plt.loglog`.
- **Reading a power law off a log-log plot.** If $y = C x^p$, then on log-log axes it's a straight
  line with slope $p$, and you can measure $p$ from two points with `np.log`.

## 0. Setup

Here's where we do all of our python setup. We'll grab matplotlib for plotting, sympy for symbolic manipulation, and numpy for numerical python calculations

In [ ]:
%matplotlib inline
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from sympy import *
init_printing()

**One warning before we start.** The line `from sympy import *` means that plain `sin`, `exp`,
and `pi` are now the *SymPy* versions, which are for symbols, not for arrays of numbers. In this
notebook we're doing numbers, so always write `np.sin`, `np.exp`, and `np.pi`. (If you get an
error that mentions `ImmutableDenseNDimArray`, or an array that prints as `[0 pi/4 pi/2]`
instead of decimals, this is why.)

## 1. Predict first (before you run anything)

**How large can $\theta$ get, in degrees, before $\sin\theta \approx \theta$ is wrong by more
than 1%?** Write your number down -- most people guess far too small.

Double-click the cell below, type your prediction, and press Shift-Enter.

*(your prediction here)*

## 2. Demo: doing math on a whole array at once

We'll practice on a *different* linear approximation: $e^x \approx 1 + x$ for small $x$.

`np.linspace(0, 1, 5)` makes an array of 5 evenly spaced numbers from 0 to 1. When you hand an
array to a NumPy function like `np.exp`, you get back an array of the same length, with the function
applied to each element. Same for arithmetic: `1 + x` adds 1 to every element.

In [ ]:
x = np.linspace(0, 1, 5)
print('x       =', x)
print('exp(x)  =', np.exp(x))
print('1 + x   =', 1 + x)

The **relative error** is how wrong the approximation is, as a fraction of the true value:

$$\text{relative error} = \frac{|\text{exact} - \text{approximation}|}{|\text{exact}|} .$$

Since everything works element-wise, this is one line, and it gives one relative error per $x$.
(`np.abs` is absolute value.) A relative error of 0.01 means "wrong by 1%".

In [ ]:
relative_error = np.abs(np.exp(x) - (1 + x)) / np.abs(np.exp(x))
print('relative error =', relative_error)

## 3. Your turn: where does $\sin\theta \approx \theta$ hit 1%?

In the cell below, change the two marked lines so that `exact` is $\sin\theta$ and `approx` is
its linear approximation $\theta$. As handed to you, the cell still does the demo's $e^x$.

Notice that `theta` starts at 0.01 rad, not at 0: at exactly $\theta = 0$, $\sin\theta = 0$, and the
relative error would be $0/0$. (Starting at 0.01 rad, about half a degree, costs us nothing.)
`np.degrees` converts radians to degrees so we can read the answer in degrees.

In [ ]:
theta = np.linspace(0.01, np.pi / 2, 400)
theta_degrees = np.degrees(theta)

# Change the line below.
exact = np.exp(theta)
# Change the line below.
approx = 1 + theta

relative_error = np.abs(exact - approx) / np.abs(exact)

Now let's plot it. The first plot shows the function and its approximation together. The second
shows the relative error, with a gray line at 1%. The `plt.xlim` and `plt.ylim` lines zoom in on the
bottom left of that plot so you can read off the crossing; delete them (or change them) to see the whole
thing.

In [ ]:
plt.plot(theta_degrees, exact, label='exact')
plt.plot(theta_degrees, approx, '--', label='linear approximation')
plt.xlabel('angle (degrees)')
plt.legend()
plt.show()

plt.plot(theta_degrees, relative_error)
plt.axhline(0.01, color='gray', label='1% error')
plt.xlim(0, 30)
plt.ylim(0, 0.03)
plt.grid(True)
plt.xlabel('angle (degrees)')
plt.ylabel('relative error')
plt.legend()
plt.show()

## 4. Demo: log-log plots and power laws

Here's a very useful trick. Suppose $y = C x^p$. Take the log of both sides:

$$\log y = \log C + p \log x .$$

That's a straight line, if you plot $\log y$ against $\log x$, and its **slope is the power $p$**.
`plt.loglog` does exactly that: it plots $y$ against $x$ with both axes on a log scale. So a power
law shows up as a straight line, and the steeper the line, the bigger the power.

Here are two power laws that are *not* the ones in your homework: $y = 5x^4$ and
$y = 2\sqrt{x} = 2x^{1/2}$.

In [ ]:
def demo_power_law(x):
    return 5 * x**4

def demo_square_root(x):
    return 2 * np.sqrt(x)

x = np.linspace(0.01, 1, 400)
plt.loglog(x, demo_power_law(x), label='5 x^4')
plt.loglog(x, demo_square_root(x), label='2 sqrt(x)')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

Both are straight lines, and the $x^4$ one is much steeper. To *measure* the slope, pick two
points on the line and do rise over run, in logs:

$$p = \frac{\log y_2 - \log y_1}{\log x_2 - \log x_1} .$$

`np.log` is the natural log. (Any log works, as long as you use the same one on top and bottom.)
The 5 out front doesn't matter: it shifts the line up and down but doesn't change its slope.

In [ ]:
x_1 = 0.1
x_2 = 0.5
y_1 = demo_power_law(x_1)
y_2 = demo_power_law(x_2)
slope = (np.log(y_2) - np.log(y_1)) / (np.log(x_2) - np.log(x_1))
print('slope =', slope)

## 5. Predict again (before you plot the error)

Next you'll plot the **absolute error** $|\sin\theta - \theta|$ against $\theta$ on log-log axes.
For small $\theta$, $\sin\theta - \theta$ is small like $\theta$ to some power $p$, so the plot
should be a straight line of slope $p$. **What slope do you predict?** One sentence on why.

Double-click the cell below, type your prediction, and press Shift-Enter.

*(your prediction here)*

## 6. Your turn: the slope of the error

Change the marked lines so that `abs_error` is $|\sin\theta - \theta|$ (an array, one value per
angle), and `error_1` and `error_2` are $|\sin\theta - \theta|$ at the two angles `theta_1` and
`theta_2`. As handed to you, the cell still uses the demo's $5x^4$.

In [ ]:
# Change the line below.
abs_error = 5 * theta**4

plt.loglog(theta, abs_error, label='absolute error')
plt.xlabel('theta (rad)')
plt.ylabel('absolute error')
plt.legend()
plt.show()

theta_1 = 0.05
theta_2 = 0.2
# Change the line below.
error_1 = 5 * theta_1**4
# Change the line below.
error_2 = 5 * theta_2**4
slope = (np.log(error_2) - np.log(error_1)) / (np.log(theta_2) - np.log(theta_1))
print('slope =', slope)

## 7. What you found

**(a)** Where does the relative error cross 1%, in degrees? How does that compare with your
prediction from section 1?

*(your answer here)*

**(b)** What slope did you measure, and how does it compare with your prediction from section 5?
In two or three sentences: what does the slope tell you about how fast the linear approximation
gets worse? (For example: if you double $\theta$, what happens to the error?)

*(your answer here)*

**(c)** $\sin\theta = \theta + (\text{correction})$. What do you think the *next* correction term
must look like? If the error is about $c\,\theta^p$, you can estimate $c$ as `error_2 / theta_2**p`
(use the scratch cell below if you like). We'll see where this comes from when we do Maclaurin
series.

In [ ]:
# scratch space

*(your answer here)*

**(d)** One thing that did not work along the way (a SymPy `sin` where you needed `np.sin`, a
divide-by-zero at $\theta = 0$, a plot that wasn't a straight line, ...).

*(your answer here)*